# M2 Plan A — Colab (Compute Units 절약, Auto-Resume)

**목표:** M2 (마감·표면) mAP 0.85+

**클래스 (2개):** 0: surface_defect_wall, 1: baseboard_defect

**전략 (빡빡하게):**
- yolov11l + imgsz 640 + batch 16 (A100) / 8 (T4)
- **Stage 1 30ep + Stage 2 10ep** (총 40ep)
- 자동 resume (끊겼다 재실행 시 자동 이어감)
- Drive autosave 5분마다

**Drive:** `surface.zip` (~826 MB) → `MyDrive/drone_inspect/`

**예상:** A100 약 25~40분, ~5 컴퓨트 유닛

## 사용법
1. `Run All` 한 번
2. 끊기면 다시 `Run All` (자동 resume, 80ep 두 번 안 돔)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
!pip install -q ultralytics onnx onnxslim onnxruntime-gpu numpy pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, shutil, zipfile
from pathlib import Path

# ★ 본인 계정 폴더 ★
DRIVE_DIR = Path('/content/drive/MyDrive/drone_inspect')
ZIP_NAME = 'surface.zip'

WORK = Path('/content/m2_plan_a')
WORK.mkdir(parents=True, exist_ok=True)

zip_path = DRIVE_DIR / ZIP_NAME
assert zip_path.exists(), f'{zip_path} 없음'
ds_dir = WORK / 'surface'
if not (ds_dir / 'images' / 'val').exists():
    if ds_dir.exists():
        shutil.rmtree(ds_dir, ignore_errors=True)
    print('Extracting...')
    with zipfile.ZipFile(zip_path) as z:
        for m in z.namelist():
            rel = m.replace('\\\\', '/')
            target = WORK / rel
            if rel.endswith('/'):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(m) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
    print(f'Done: {ds_dir}')
for s in ['train', 'val', 'test']:
    p = ds_dir / 'images' / s
    if p.exists():
        print(f'  {s}: {len(list(p.glob("*")))} images')

In [ ]:
yaml_text = f'''path: {ds_dir}
train: images/train
val: images/val
test: images/test

nc: 2
names:
  0: surface_defect_wall
  1: baseboard_defect
'''
data_yaml = WORK / 'surface.yaml'
data_yaml.write_text(yaml_text)
print(data_yaml.read_text())

## Drive 자동 저장 (5분마다 last.pt / best.pt 백업)

이 셀이 백그라운드 스레드를 띄워서 Stage 1/2 진행 중인 weights를 Drive에 자동 백업합니다. Colab이 끊겨도 Drive에 보존되어 다음 실행 시 자동 resume.

In [ ]:
AUTOSAVE = DRIVE_DIR / 'm2_plan_a_autosave'
AUTOSAVE.mkdir(parents=True, exist_ok=True)
PROJECT = Path('/content/runs/m2_plan_a')
PROJECT.mkdir(parents=True, exist_ok=True)

import time, threading
stop_flag = [False]
def autosave():
    while not stop_flag[0]:
        time.sleep(300)
        for s in ['stage1', 'stage2']:
            for src in [PROJECT/s/'weights/last.pt', PROJECT/s/'weights/best.pt']:
                if src.exists():
                    try:
                        shutil.copy2(src, AUTOSAVE / f'{s}_{src.name}')
                    except Exception:
                        pass
        print(f'[autosave] {time.strftime("%H:%M:%S")}')
threading.Thread(target=autosave, daemon=True).start()

## Stage 1: yolov11l + 30ep (자동 Resume 감지)

**중요**: 이 셀이 자동으로 판단합니다.
- autosave에 `stage1_last.pt`가 있으면 → **Resume** (이전 ep부터 이어감)
- 없으면 → **Fresh** (yolov11l.pt에서 새로 시작)

Colab 끊겨서 재실행해도 30ep 두 번 돌리는 일 없이 자동 이어감.

In [ ]:
from ultralytics import YOLO

stage1_dir = PROJECT / 'stage1'
stage1_weights = stage1_dir / 'weights'
stage1_weights.mkdir(parents=True, exist_ok=True)

autosave_last = AUTOSAVE / 'stage1_last.pt'
autosave_best = AUTOSAVE / 'stage1_best.pt'
local_last = stage1_weights / 'last.pt'
local_best = stage1_weights / 'best.pt'

# 자동 Resume 감지
is_resume = False
if autosave_last.exists():
    print(f'=== Stage 1 RESUME (autosave/stage1_last.pt {autosave_last.stat().st_size/1024/1024:.1f}MB 발견) ===')
    shutil.copy2(autosave_last, local_last)
    if autosave_best.exists():
        shutil.copy2(autosave_best, local_best)
    is_resume = True
else:
    print('=== Stage 1 FRESH (autosave 없음, yolov11l.pt 부터 새로 시작) ===')

if is_resume:
    # ultralytics가 last.pt에 저장된 args.yaml로부터 자동 resume
    model_s1 = YOLO(str(local_last))
    try:
        model_s1.train(resume=True)
    except Exception as e:
        # 만약 resume 실패 (이미 완료된 학습이거나 args 누락) → fresh fallback
        print(f'[WARN] Resume 실패 ({e}) — Fresh로 진행')
        is_resume = False

if not is_resume:
    print('=== Stage 1 (yolov11l + 640 + 30ep + batch=16) ===')
    model_s1 = YOLO('yolo11l.pt')
    model_s1.train(
        data=str(data_yaml),
        epochs=30,
        batch=16,
        imgsz=640,
        cache='disk',
        workers=4,
        optimizer='AdamW',
        lr0=1e-3,
        lrf=0.01,
        cos_lr=True,
        patience=10,
        warmup_epochs=3,
        close_mosaic=15,
        freeze=0,
        hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
        degrees=5.0, translate=0.1, scale=0.5,
        shear=2.0, perspective=0.001,
        flipud=0.0, fliplr=0.5,
        mosaic=1.0, mixup=0.15, copy_paste=0.5,
        erasing=0.0,
        multi_scale=0.15,
        save_period=5,
        plots=True,
        project=str(PROJECT),
        name='stage1',
        exist_ok=True
    )
print('\nStage 1 done.')

## Stage 2: fine-tune lr=1e-5 + freeze=10 + 10ep (자동 Resume 감지)

In [ ]:
stage2_dir = PROJECT / 'stage2'
stage2_weights = stage2_dir / 'weights'
stage2_weights.mkdir(parents=True, exist_ok=True)

autosave_s2_last = AUTOSAVE / 'stage2_last.pt'
autosave_s2_best = AUTOSAVE / 'stage2_best.pt'
local_s2_last = stage2_weights / 'last.pt'
local_s2_best = stage2_weights / 'best.pt'

stage1_best = PROJECT / 'stage1' / 'weights' / 'best.pt'
if not stage1_best.exists():
    raise RuntimeError(f'Stage 1 best.pt 없음: {stage1_best} — Stage 1 셀 먼저 실행')

# 자동 Resume 감지
is_resume_s2 = False
if autosave_s2_last.exists():
    print(f'=== Stage 2 RESUME (autosave/stage2_last.pt 발견) ===')
    shutil.copy2(autosave_s2_last, local_s2_last)
    if autosave_s2_best.exists():
        shutil.copy2(autosave_s2_best, local_s2_best)
    is_resume_s2 = True
else:
    print('=== Stage 2 FRESH (Stage 1 best.pt에서 fine-tune 시작) ===')

if is_resume_s2:
    model_s2 = YOLO(str(local_s2_last))
    try:
        model_s2.train(resume=True)
    except Exception as e:
        print(f'[WARN] Stage 2 Resume 실패 ({e}) — Fresh로 진행')
        is_resume_s2 = False

if not is_resume_s2:
    print(f'=== Stage 2 fine-tune (lr=1e-5, freeze=10, 10ep) ===')
    model_s2 = YOLO(str(stage1_best))
    model_s2.train(
        data=str(data_yaml),
        epochs=10,
        batch=16,
        imgsz=640,
        cache='disk',
        workers=4,
        optimizer='AdamW',
        lr0=1e-5,
        lrf=0.01,
        cos_lr=True,
        patience=6,
        warmup_epochs=1,
        close_mosaic=8,
        freeze=10,
        mosaic=0.5, mixup=0.0, copy_paste=0.2,
        save_period=5,
        plots=True,
        project=str(PROJECT),
        name='stage2',
        exist_ok=True
    )
stop_flag[0] = True  # autosave 종료
print('\nStage 2 done.')

## ONNX Export + 평가 + Drive 저장

In [ ]:
stage1_best = PROJECT / 'stage1' / 'weights' / 'best.pt'
stage2_best = PROJECT / 'stage2' / 'weights' / 'best.pt'

# Stage 1 vs Stage 2 mAP 비교
def best_map(csv_path):
    if not csv_path.exists():
        return -1
    best = -1
    with open(csv_path) as f:
        for i, line in enumerate(f):
            if i == 0: continue
            parts = line.strip().split(',')
            if len(parts) < 8: continue
            try: best = max(best, float(parts[7]))
            except: pass
    return best

s1_map = best_map(PROJECT / 'stage1' / 'results.csv')
s2_map = best_map(PROJECT / 'stage2' / 'results.csv')
print(f'Stage 1 mAP50: {s1_map:.4f}')
print(f'Stage 2 mAP50: {s2_map:.4f}')

best_path = stage2_best if (stage2_best.exists() and s2_map >= s1_map) else stage1_best
print(f'채택: {best_path}')

best_model = YOLO(str(best_path))
best_model.export(format='onnx', opset=17, dynamic=True, simplify=True)
onnx_path = best_path.with_suffix('.onnx')

metrics = best_model.val(data=str(data_yaml), imgsz=640, batch=16, split='test')
print(f'\n=== M2 최종 ===')
print(f'  mAP50:    {metrics.box.map50:.4f}')
print(f'  mAP50-95: {metrics.box.map:.4f}')
print(f'  precision: {metrics.box.mp:.4f}')
print(f'  recall:    {metrics.box.mr:.4f}')
print(f'  0.85+? {"YES" if metrics.box.map50 >= 0.85 else "NO"}')

# Drive 저장
OUT = DRIVE_DIR / 'm2_plan_a_results'
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(onnx_path, OUT / 'm2_yolo_surface.onnx')
shutil.copy2(best_path, OUT / 'm2_yolo_surface_best.pt')
for s in ['stage1', 'stage2']:
    rcsv = PROJECT / s / 'results.csv'
    if rcsv.exists():
        shutil.copy2(rcsv, OUT / f'{s}_results.csv')
print(f'\nSaved to: {OUT}')